In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
from skimage.filters.rank import entropy
from skimage.morphology import disk
from skimage.color import rgb2gray
from skimage.util import img_as_ubyte
import matplotlib.pyplot as plt
import seaborn as sns


STEP 1: Image Acquisition & Preprocessing

In [ ]:
def preprocess_image(image):
    # Resize image for consistency
    img_resized = cv2.resize(image, (256, 256))
    
    # Convert to HSV for color-based enhancement
    hsv = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)
    
    # Histogram equalization on V channel to enhance contrast
    h, s, v = cv2.split(hsv)
    v_eq = cv2.equalizeHist(v)
    hsv_eq = cv2.merge([h, s, v_eq])
    img_enhanced = cv2.cvtColor(hsv_eq, cv2.COLOR_HSV2BGR)
    
    # Gaussian blur to reduce noise
    img_denoised = cv2.GaussianBlur(img_enhanced, (5, 5), 0)
    
    return img_denoised


STEP2:  Image Segmentation

In [ ]:
def segment_diseased_region(image):
    # Convert to HSV and create mask for yellowish color (typical of disease spots)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lower = np.array([10, 40, 40])
    upper = np.array([40, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)
    result = cv2.bitwise_and(image, image, mask=mask)
    return image, mask, result



In [ ]:
folder = r"C:\Users\Carl Renz\OneDrive\Desktop\DIP_FN\plant-test\test"

for filename in os.listdir(folder):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        if img is None:
            print(f"Could not read image: {img_path}")
            continue
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)

# Optional: Morphological operation to clean up the threshold
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))
img_morph = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

contours, _ = cv2.findContours(img_morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print(f"Contours found: {len(contours)}")

if len(contours) == 0:
    print("No contours found!")

cv2.imshow("Threshold", thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


STEP 3: Feature Extraction

In [ ]:
def extract_features(image, mask):
    features = {}
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Avoid empty mask to prevent errors
    if np.count_nonzero(mask) == 0:
        # No diseased area detected, fill with zeros
        features['mean_hue'] = 0
        features['mean_saturation'] = 0
        features['mean_value'] = 0
        features['mean_entropy'] = 0
        features['area'] = 0
        features['perimeter'] = 0
        features['solidity'] = 0
        return features
    
    # Color features within diseased region
    features['mean_hue'] = np.mean(hsv[:,:,0][mask > 0])
    features['mean_saturation'] = np.mean(hsv[:,:,1][mask > 0])
    features['mean_value'] = np.mean(hsv[:,:,2][mask > 0])
    
    # Entropy (texture) within diseased region
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    entropy_img = entropy(img_as_ubyte(gray), disk(5))
    features['mean_entropy'] = np.mean(entropy_img[mask > 0])
    
    # Shape features from contours in mask
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(c)
        perimeter = cv2.arcLength(c, True)
        hull = cv2.convexHull(c)
        hull_area = cv2.contourArea(hull)
        solidity = area / hull_area if hull_area > 0 else 0
        features['area'] = area
        features['perimeter'] = perimeter
        features['solidity'] = solidity
    else:
        features['area'] = 0
        features['perimeter'] = 0
        features['solidity'] = 0
        
    return features

STEP 4:  Post Classification

In [ ]:
# Step 1: Load the CSV file
df = pd.read_csv('results.csv')

# Step 2: Correct the true label using filename
def get_true_label(filename):
    filename = filename.lower()
    if 'healthy' in filename:
        return 'Healthy'
    else:
        return 'Diseased'

df['disease_status'] = df['filename'].apply(get_true_label)

# Step 3: Define the rule-based classifier
def classify_disease(features):
    if (3.6 <= features['entropy'] <= 4.75 and
        65 <= features['mean_red'] <= 195 and
        38000 <= features['area'] <= 51000):
        return "Healthy"
    else:
        return "Diseased"

# Step 4: Apply the classifier to each row
df['predicted_status'] = df.apply(classify_disease, axis=1)

# Step 5: Evaluate the classification
print("\nSample Predictions:")
print(df[['filename', 'disease_status', 'predicted_status']].head(10))

accuracy = (df['disease_status'] == df['predicted_status']).mean()
print(f"\nClassification Accuracy: {accuracy:.2%}")

conf_matrix = pd.crosstab(df['disease_status'], df['predicted_status'], rownames=['Actual'], colnames=['Predicted'])
print("\nConfusion Matrix:")
print(conf_matrix)

STEP 5:Disease classification

In [ ]:
def process_folder(folder_path, show_images=False, save_csv=True):
    results = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.jpg', '.png', '.jpeg')):
            print(f"Processing file: {filename}")  # Debug print
            img_path = os.path.join(folder_path, filename)

            img = cv2.imread(img_path)
            if img is None:
                print(f"Failed to load {filename}")
                continue

            # ======= 1. Preprocessing =======
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_blur = cv2.GaussianBlur(img_rgb, (5, 5), 0)
            img_gray = cv2.cvtColor(img_blur, cv2.COLOR_RGB2GRAY)
            img_eq = cv2.equalizeHist(img_gray)

            # ======= 2. Segmentation =======
            _, img_thresh = cv2.threshold(img_eq, 127, 255, cv2.THRESH_BINARY_INV)
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
            img_morph = cv2.morphologyEx(img_thresh, cv2.MORPH_CLOSE, kernel)

            # ======= 3. Feature Extraction =======
            contours, _ = cv2.findContours(img_morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if len(contours) == 0:
                print(f"No contours found for {filename}")
                continue
            else:
                print(f"Found {len(contours)} contours for {filename}")

            largest_contour = max(contours, key=cv2.contourArea)
            area = cv2.contourArea(largest_contour)
            perimeter = cv2.arcLength(largest_contour, True)

            entropy_val = entropy(img_as_ubyte(img_eq), disk(5)).mean()

            mask = np.zeros(img_morph.shape, dtype=np.uint8)
            cv2.drawContours(mask, [largest_contour], -1, 255, thickness=cv2.FILLED)
            mean_color = cv2.mean(img_rgb, mask=mask)  # (R, G, B, alpha)

            print(f"Extracted features for {filename}: area={area}, perimeter={perimeter}, entropy={entropy_val}, mean_red={mean_color[0]}")

            features = {
                'filename': filename,
                'area': area,
                'perimeter': perimeter,
                'entropy': entropy_val,
                'mean_red': mean_color[0],
                'mean_green': mean_color[1],
                'mean_blue': mean_color[2],
            }

            # Classify disease
            features['disease_status'] = classify_disease(features)

            print("Features dict with classification:", features)

            results.append(features)

            if show_images:
                cv2.imshow('Original', img)
                cv2.imshow('Segmented', img_morph)
                cv2.waitKey(500)
                cv2.destroyAllWindows()

    if save_csv:
        df = pd.DataFrame(results)
        print(f"Saving CSV with {len(results)} entries.")
        df.to_csv('results.csv', index=False)
        print("CSV saved successfully.")

In [ ]:

dataset_folder = r"C:\Users\Carl Renz\OneDrive\Desktop\DIP_FN\plant-test\test"
process_folder(dataset_folder, show_images=False, save_csv=True)


Generating Summary Statistics from  CSV results

In [ ]:

# Example: load your results CSV into a DataFrame
df = pd.read_csv('results.csv')

# Print first few rows to inspect
print(df.head())

# Check column types to see which are numeric
print(df.dtypes)

# Count of samples per disease status (class distribution)
class_counts = df['disease_status'].value_counts()
print("\nClass distribution:")
print(class_counts)

# Select numeric columns only for aggregation
numeric_cols = df.select_dtypes(include='number').columns

# Group by disease_status and calculate mean and std only for numeric columns
group_stats = df.groupby('disease_status')[numeric_cols].agg(['mean', 'std'])
print("\nMean and StdDev of numeric features per class:")
print(group_stats)

# Optionally, save group stats to CSV for reporting
group_stats.to_csv('group_stats_summary.csv')

# You can now refer to these stats in your Results and Discussion sections


GENERATING VISUALIZATION (bar chart & box plots)

In [ ]:
df = pd.read_csv('results.csv')

# Features to visualize
# 1) Pairwise scatter plot
features = ['area', 'perimeter', 'entropy', 'mean_red', 'mean_green', 'mean_blue']

sns.pairplot(df, vars=features, hue='disease_status', diag_kind='kde', markers=['o', 's'], palette='Set1')
plt.suptitle('Pairwise Scatter Plots of Features by Disease Status', y=1.02)
plt.show()

features = ['area', 'perimeter', 'entropy', 'mean_red', 'mean_green', 'mean_blue']

# 2) Bar chart: Mean feature values by disease status
mean_features = df.groupby('disease_status')[features].mean().reset_index()

plt.figure(figsize=(12, 6))
mean_features_melted = mean_features.melt(id_vars='disease_status', var_name='Feature', value_name='Mean Value')

sns.barplot(data=mean_features_melted, x='Feature', y='Mean Value', hue='disease_status')
plt.title('Mean Feature Values by Disease Status')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 3) Box plots: Distribution of features by disease status
plt.figure(figsize=(15, 10))

for i, feature in enumerate(features, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(data=df, x='disease_status', y=feature)
    plt.title(f'{feature} by Disease Status')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate summary statistics (mean and std) by disease status
summary_stats = df.groupby('disease_status')[features].agg(['mean', 'std'])

# Convert to Markdown format
markdown_table = summary_stats.to_markdown()

# Save to a markdown file
with open('summary_statistics.md', 'w') as f:
    f.write(markdown_table)

print("Markdown table saved to summary_statistics.md")

In [ ]:
total = len(df)
healthy = class_counts.get('Healthy', 0)
diseased = class_counts.get('Diseased', 0)
healthy_pct = round(healthy / total * 100, 2)
diseased_pct = round(diseased / total * 100, 2)

entropy_healthy = df[df['disease_status']=='Healthy']['entropy'].mean()
entropy_diseased = df[df['disease_status']=='Diseased']['entropy'].mean()
area_healthy = df[df['disease_status']=='Healthy']['area'].mean()
area_diseased = df[df['disease_status']=='Diseased']['area'].mean()

print(f"""
A total of {total} leaf images were processed. The classifier categorized {healthy} samples ({healthy_pct}%) as Healthy and {diseased} samples ({diseased_pct}%) as Diseased.

Analysis showed diseased leaves have a higher mean entropy ({entropy_diseased:.2f}) compared to healthy leaves ({entropy_healthy:.2f}). Mean leaf area was {area_diseased:.2f} pixels for diseased and {area_healthy:.2f} pixels for healthy samples.
""")
